# Task 1: Sentence Transformer Implementation

## Objective
To implement a sentence transformer model that converts input sentences into fixed-length embeddings using a pre-trained transformer model.

## Model Architecture and Implementation
I used the `bert-base-uncased` model from Hugging Face's `transformers` library as the backbone of our sentence transformer. The implementation consists of the following steps:

### 1. **Tokenizer and Model Loading**
- I initialize a BERT tokenizer and model using `BertTokenizer.from_pretrained` and `BertModel.from_pretrained`.
- These components handle tokenization and encoding of input sentences respectively.

### 2. **Mean Pooling Layer**
- BERT produces embeddings for each token in the sentence.
- I implemented a custom `mean_pooling` function to convert these variable-length token embeddings into a single fixed-length vector per sentence.
- This is done by computing the weighted average of the token embeddings, ignoring the padding tokens using the attention mask.

### 3. **Forward Pass**
- The forward method takes a list of sentences, tokenizes them, feeds them into the BERT model, and applies mean pooling to produce sentence-level embeddings.
- Gradients are disabled during inference (`torch.no_grad()`), as I am only testing the model, not training it.

### 4. **Sample Usage**
- Two sample sentences are passed through the model.
- Their embeddings are printed to verify that the model is functioning correctly.

## Key Decisions and Justification
- **BERT Base Uncased**: Chosen for its balance between performance and efficiency.
- **Mean Pooling**: Offers more robust representations than using just the [CLS] token, especially in downstream tasks.
- **No Fine-Tuning**: For Task 1, I used frozen BERT weights as I am focused on testing sentence encoding, not classification or task-specific training.

## Result
- The model outputs dense vectors (typically 768 dimensions) for each sentence, which can be used in similarity tasks or as input to downstream classifiers.

In [1]:
import torch
from torch import nn
from transformers import BertTokenizer, BertModel

class SentenceTransformer(nn.Module):
    def __init__(self, model_name='bert-base-uncased'):
        super(SentenceTransformer, self).__init__()
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert = BertModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def forward(self, sentences):
        encoded_input = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
        with torch.no_grad():
            model_output = self.bert(**encoded_input)
        sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
        return sentence_embeddings

if __name__ == '__main__':
    model = SentenceTransformer()
    sentences = ["The sky is blue.", "The dog is running."]
    embeddings = model(sentences)
    print("Sentence Embeddings:")
    print(embeddings)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Sentence Embeddings:
tensor([[ 0.4448, -0.2485, -0.0890,  ...,  0.0019,  0.3094, -0.0130],
        [ 0.0126,  0.1429, -0.0595,  ..., -0.2325,  0.0732, -0.0531]])


# Task 2: Multi-Task Learning Expansion

## Objective
To expand the sentence transformer model to support multiple NLP tasks in a multi-task learning (MTL) setting.

### Task A: Sentence Classification
- I added a classification head to assign sentences into categories.
- Example classes: ["sports", "politics", "technology"]

### Task B: Sentiment Analysis
- I added another head to perform binary sentiment analysis (positive/negative).

## Model Architecture and Changes
To support MTL, I created a new model class with the following components:
- A shared BERT encoder.
- A task-specific linear layer for sentence classification.
- A task-specific linear layer for sentiment analysis.

## Key Decisions and Justification
- **Task Selection**: I chose sentence classification and sentiment analysis due to their relevance and differing output structures.
- **Shared Encoder**: I reused the BERT encoder across both tasks to enable shared feature learning.
- **Separate Heads**: I used independent linear heads for each task, allowing specialized learning without interference.

## Result
- The model now produces two outputs per input sentence: a category label and a sentiment score.
- This architecture supports simultaneous learning of both tasks during training by combining their respective loss functions.

The architecture is shown below:

In [2]:
class MultiTaskSentenceTransformer(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_classes_a=3, num_classes_b=2):
        super(MultiTaskSentenceTransformer, self).__init__()
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.bert = BertModel.from_pretrained(model_name)
        self.classifier_a = nn.Linear(self.bert.config.hidden_size, num_classes_a)  # Task A
        self.classifier_b = nn.Linear(self.bert.config.hidden_size, num_classes_b)  # Task B

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def forward(self, sentences):
        encoded_input = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
        model_output = self.bert(**encoded_input)
        pooled_output = self.mean_pooling(model_output, encoded_input['attention_mask'])
        output_a = self.classifier_a(pooled_output)
        output_b = self.classifier_b(pooled_output)
        return {"task_a": output_a, "task_b": output_b}

if __name__ == '__main__':
    model = MultiTaskSentenceTransformer()
    sentences = ["Apple releases new iPhone.", "The team won the championship!"]
    outputs = model(sentences)
    print("Task A (Classification) Output:", outputs['task_a'])
    print("Task B (Sentiment) Output:", outputs['task_b'])

Task A (Classification) Output: tensor([[ 0.2065, -0.0251, -0.3166],
        [ 0.0596, -0.1909,  0.0612]], grad_fn=<AddmmBackward0>)
Task B (Sentiment) Output: tensor([[ 0.0021, -0.0179],
        [ 0.0856, -0.0183]], grad_fn=<AddmmBackward0>)


# Task 3: Training Considerations

## Scenario Analysis

### 1. Freezing the Entire Network
- **Implication**: No part of the model is updated during training.
- **Advantage**: Fastest inference and avoids overfitting. Suitable for scenarios where computational resources are limited.
- **Use Case**: I would freeze the entire model if I wanted to use it only for feature extraction or zero-shot inference, or if I had extremely limited or no task-specific labeled data.

### 2. Freezing Only the Transformer Backbone
- **Implication**: Only the task-specific heads are trainable. The shared encoder remains unchanged.
- **Advantage**: Efficient training that allows specialization for new tasks while leveraging pre-trained knowledge.
- **Use Case**: I would freeze the transformer backbone when I have a small dataset and want to train lightweight task-specific classifiers without affecting the general language representations learned by the transformer.

### 3. Freezing One of the Task Heads
- **Implication**: Only one task-specific head is kept static while the encoder and the other head can adapt.
- **Advantage**: Supports continual learning and avoids catastrophic forgetting in already well-trained tasks.
- **Use Case**: I would freeze one task head when it has already been optimized and validated, and I want to train the model further on a new task or domain without degrading the performance of the original task.

## Transfer Learning Approach

### 1. Choice of Pre-trained Model
- I would choose a pre-trained language model like `bert-base-uncased` for general tasks. For domain-specific applications, I would opt for specialized models such as `BioBERT`, `SciBERT`, or `LegalBERT`.

### 2. Layers to Freeze/Unfreeze
- Initially, I would freeze most of the transformer layers to retain general-purpose language understanding.
- I would unfreeze only the top layers of the encoder and the task-specific heads to allow for fine-tuning on the target tasks.
- Optionally, I might gradually unfreeze more layers during training (e.g., using a technique like gradual unfreezing).

### 3. Rationale
- Freezing early layers helps preserve the fundamental linguistic knowledge learned during pretraining.
- Unfreezing higher layers and heads gives the model the flexibility to adapt to new task-specific features.
- This approach balances efficient training with strong performance, especially in low-resource settings.

# Task 4: Training Loop Implementation (BONUS)

## Objective
To outline a hypothetical training loop for the multi-task sentence transformer model that handles both sentence classification and sentiment analysis.

## Assumptions
- I assume we have a synthetic or hypothetical dataset where each input sentence has two associated labels:
  - One for **sentence classification** (Task A)
  - One for **sentiment analysis** (Task B)
- Both tasks are framed as classification problems using `CrossEntropyLoss`.
- Labels and sentences are tokenized in advance or within the dataset class.
- For simplicity, I assume equal importance between the two tasks during training.

## Multi-Task Learning Training Framework
Multi-task learning (MTL) allows a model to learn shared representations that benefit more than one objective.
In this loop, I:
- Perform a **forward pass** through a shared transformer backbone.
- Route the pooled sentence representation to two different **task-specific heads**.
- Compute two **loss values** (one per task) and combine them.
- Backpropagate the total loss and update the model.

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

# Hypothetical dataset class
class DummyMultiTaskDataset(Dataset):
    def __init__(self):
        self.sentences = ["Example sentence one", "Another sentence"]
        self.labels_a = [0, 1]  # Sentence classification labels
        self.labels_b = [1, 0]  # Sentiment labels

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return self.sentences[idx], torch.tensor(self.labels_a[idx]), torch.tensor(self.labels_b[idx])

# Dataloader setup
train_dataset = DummyMultiTaskDataset()
dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# Assume: model already initialized
model.train()
optimizer = Adam(model.parameters(), lr=2e-5)
criterion = CrossEntropyLoss()
num_epochs = 3

for epoch in range(num_epochs):
    total_loss = 0
    correct_a = correct_b = total = 0

    for batch in dataloader:
        sentences, labels_a, labels_b = batch

        outputs = model(sentences)
        logits_a = outputs['task_a']
        logits_b = outputs['task_b']

        loss_a = criterion(logits_a, labels_a)
        loss_b = criterion(logits_b, labels_b)
        loss = loss_a + loss_b

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total += labels_a.size(0)

        correct_a += (logits_a.argmax(dim=1) == labels_a).sum().item()
        correct_b += (logits_b.argmax(dim=1) == labels_b).sum().item()

    acc_a = 100 * correct_a / total
    acc_b = 100 * correct_b / total
    print(f"Epoch {epoch + 1}: Loss = {total_loss:.4f}, AccA = {acc_a:.2f}%, AccB = {acc_b:.2f}%")

Epoch 1: Loss = 1.7430, AccA = 0.00%, AccB = 50.00%
Epoch 2: Loss = 1.4923, AccA = 100.00%, AccB = 50.00%
Epoch 3: Loss = 1.2324, AccA = 100.00%, AccB = 100.00%


## Decisions and Insights
- I used `CrossEntropyLoss` for both heads assuming both are classification tasks.
- I summed the two losses equally. In real-world settings, I could introduce task-specific weights or learnable loss scaling.
- I computed accuracy for both tasks to track task-specific progress.
- I used a shared optimizer to update all trainable parameters jointly.

## Notes on Metrics
- **Accuracy** is sufficient for demonstration, but in real deployments:
  - For classification tasks with class imbalance, I would prefer **macro F1-score**.
  - For sentiment analysis, **precision/recall** breakdown can be valuable.

## Summary
This training setup demonstrates how shared representations can support multiple NLP tasks in a single pass. The structure is extensible to more complex objectives, and metrics or weighting strategies can be adapted depending on domain and task difficulty.